# 📊 Notebook 04 – Model Evaluation & Benchmarking

**SpaceSLM-mini** | Evaluation Step

This notebook compares three models:
1. **Base model** – Qwen2.5-0.5B-Instruct (pretrained, no fine-tuning)
2. **Fine-tuned model** – after QLoRA on space domain data
3. **Quantized model** – Q4_K_M GGUF (local CPU)

---

### 🎯 What you will measure
- Answer quality (ROUGE-L score)
- Model size on disk (MB)
- Peak RAM usage (MB)
- Inference latency (seconds/response)
- Throughput (tokens/second)

### 📚 Key concepts
- **ROUGE-L**: Measures overlap between generated and reference text using longest common subsequences. 0 = no match, 1 = perfect match.
- **Quantization impact**: How much quality do we lose when compressing to 4-bit?
- **Domain adaptation**: Does fine-tuning actually improve space-domain answers?


In [ ]:
import json, time, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Install if needed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rouge-score', 'psutil', '-q'])
from rouge_score import rouge_scorer

print('✅ Imports OK')

In [ ]:
# Fixed benchmark questions (same as benchmark.py)
BENCHMARK_QA = [
    {'category': 'factual',
     'instruction': 'What is a geostationary orbit and why is it used for communication satellites?',
     'reference': 'A geostationary orbit is a circular orbit at approximately 35,786 km altitude above the equator where a satellite orbital period matches Earths rotation, keeping it stationary relative to the ground, making it ideal for continuous communication coverage over a fixed area.'},
    {'category': 'subsystem',
     'instruction': 'Explain the purpose of a satellite power subsystem.',
     'reference': 'A satellite power subsystem generates, stores, and distributes electrical energy to onboard components, typically using solar panels to generate power and batteries to store energy during eclipse periods.'},
    {'category': 'subsystem',
     'instruction': 'What does the thermal control subsystem do on a spacecraft?',
     'reference': 'The thermal control subsystem maintains all spacecraft components within their operating temperature ranges using passive methods like coatings and insulation, and active methods like heaters and heat pipes.'},
    {'category': 'terminology',
     'instruction': 'What is delta-v in the context of spacecraft propulsion?',
     'reference': 'Delta-v is a measure of the change in velocity a spacecraft can achieve using its propulsion system, used to characterise the amount of fuel required for orbital manoeuvres.'},
    {'category': 'factual',
     'instruction': 'What is the purpose of a satellite attitude control system?',
     'reference': 'An attitude control system maintains or changes the orientation of a spacecraft in space using reaction wheels, magnetorquers, or thrusters to point antennas, solar panels, or scientific instruments in the correct direction.'},
]

print(f'Benchmark: {len(BENCHMARK_QA)} questions across {len(set(q["category"] for q in BENCHMARK_QA))} categories')

In [ ]:
# Shared utilities
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_l(pred: str, ref: str) -> float:
    return scorer.score(ref, pred)['rougeL'].fmeasure

def build_prompt(instruction: str) -> str:
    return f'### Instruction:\n{instruction}\n\n### Response:\n'

def get_dir_size_mb(path: str) -> float:
    p = Path(path)
    if p.is_file(): return p.stat().st_size / 1e6
    if p.is_dir():  return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
    return 0.0

print('✅ Utilities ready')

In [ ]:
# ── Evaluate a HuggingFace model ──────────────────────────────────────────────
def eval_hf_model(model_path: str, label: str) -> dict:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import psutil, os
    
    print(f'\n📦 Loading [{label}]: {model_path}')
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True)
    model.eval()
    
    ram_mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
    latencies, rouge_scores, tok_counts = [], [], []
    
    for qa in BENCHMARK_QA:
        prompt = build_prompt(qa['instruction'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200,
                                 temperature=0.1, do_sample=True,
                                 pad_token_id=tokenizer.eos_token_id)
        lat = time.perf_counter() - t0
        new_toks = out[0][inputs['input_ids'].shape[1]:]
        pred = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
        latencies.append(lat)
        rouge_scores.append(rouge_l(pred, qa['reference']))
        tok_counts.append(len(new_toks))
    
    return {
        'model':          label,
        'size_mb':        get_dir_size_mb(model_path),
        'peak_ram_mb':    ram_mb,
        'mean_latency_s': sum(latencies) / len(latencies),
        'tokens_per_sec': sum(tok_counts) / sum(latencies),
        'mean_rouge_l':   sum(rouge_scores) / len(rouge_scores),
    }

print('Function defined')

In [ ]:
results = []

# ── 1. Base model ─────────────────────────────────────────────────────────────
try:
    r = eval_hf_model('Qwen/Qwen2.5-0.5B-Instruct', 'Base (pretrained)')
    results.append(r)
    print(f"  ROUGE-L: {r['mean_rouge_l']:.4f}  |  {r['tokens_per_sec']:.1f} tok/s")
except Exception as e:
    print(f'Base model eval failed: {e}')

# ── 2. Fine-tuned model ───────────────────────────────────────────────────────
if Path('models/merged_fp16').exists():
    try:
        r = eval_hf_model('models/merged_fp16', 'Fine-tuned FP16')
        results.append(r)
        print(f"  ROUGE-L: {r['mean_rouge_l']:.4f}  |  {r['tokens_per_sec']:.1f} tok/s")
    except Exception as e:
        print(f'Fine-tuned model eval failed: {e}')
else:
    print('Fine-tuned model not found – run notebook 03 first.')

In [ ]:
# ── 3. GGUF model ─────────────────────────────────────────────────────────────
GGUF_PATH = 'models/gguf/space-slm-q4_k_m.gguf'

if Path(GGUF_PATH).exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'llama-cpp-python', '-q'])
    from llama_cpp import Llama
    import psutil, os
    
    print(f'Loading GGUF: {GGUF_PATH}')
    llm = Llama(model_path=GGUF_PATH, n_ctx=512, n_threads=4, verbose=False)
    ram_mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
    
    latencies, rouge_scores, tok_counts = [], [], []
    for qa in BENCHMARK_QA:
        prompt = build_prompt(qa['instruction'])
        t0 = time.perf_counter()
        out = llm(prompt, max_tokens=200, temperature=0.1,
                  stop=['### Instruction:'], echo=False)
        lat = time.perf_counter() - t0
        pred = out['choices'][0]['text'].strip()
        latencies.append(lat)
        rouge_scores.append(rouge_l(pred, qa['reference']))
        tok_counts.append(out['usage']['completion_tokens'])
    
    r = {'model': 'GGUF Q4_K_M',
         'size_mb': get_dir_size_mb(GGUF_PATH),
         'peak_ram_mb': ram_mb,
         'mean_latency_s': sum(latencies)/len(latencies),
         'tokens_per_sec': sum(tok_counts)/sum(latencies),
         'mean_rouge_l': sum(rouge_scores)/len(rouge_scores)}
    results.append(r)
    print(f"GGUF ROUGE-L: {r['mean_rouge_l']:.4f}  |  {r['tokens_per_sec']:.1f} tok/s")
else:
    print(f'GGUF model not found at {GGUF_PATH} – run convert_gguf.sh first.')

In [ ]:
# ── Results Table ─────────────────────────────────────────────────────────────
if results:
    df = pd.DataFrame(results)
    df = df.round(3)
    print('\n📊 Benchmark Results:')
    print(df.to_string(index=False))
    
    Path('results').mkdir(exist_ok=True)
    df.to_csv('results/benchmark.csv', index=False)
    print('\n✅ Saved to results/benchmark.csv')

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
if results and len(results) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    df = pd.DataFrame(results)
    
    # ROUGE-L comparison
    axes[0].bar(df['model'], df['mean_rouge_l'], color=['#4285F4','#34A853','#EA4335'])
    axes[0].set_title('Answer Quality (ROUGE-L)', fontweight='bold')
    axes[0].set_ylabel('ROUGE-L Score (higher = better)')
    axes[0].tick_params(axis='x', rotation=15)
    
    # Tokens per second
    axes[1].bar(df['model'], df['tokens_per_sec'], color=['#4285F4','#34A853','#EA4335'])
    axes[1].set_title('Inference Speed (tokens/sec)', fontweight='bold')
    axes[1].set_ylabel('tokens/sec (higher = faster)')
    axes[1].tick_params(axis='x', rotation=15)
    
    # Model size
    axes[2].bar(df['model'], df['size_mb'], color=['#4285F4','#34A853','#EA4335'])
    axes[2].set_title('Model Size (MB)', fontweight='bold')
    axes[2].set_ylabel('Size in MB (lower = smaller)')
    axes[2].tick_params(axis='x', rotation=15)
    
    plt.suptitle('SpaceSLM-mini – Model Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    Path('results').mkdir(exist_ok=True)
    plt.savefig('results/benchmark_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to results/benchmark_chart.png')

## ✅ Evaluation Complete!

Your benchmark results are saved in `results/benchmark.csv`.

### Interpreting the results

| Observation | Meaning |
|-------------|----------|
| Fine-tuned ROUGE-L > Base ROUGE-L | Domain adaptation worked ✅ |
| GGUF ROUGE-L ≈ Fine-tuned ROUGE-L | Quantization preserved quality ✅ |
| GGUF smaller size | Quantization reduced storage ✅ |
| GGUF lower latency | Efficient CPU inference ✅ |
